# Electoral Gender Quotas and Democratic Legitimacy

**Authors:** Amanda Clayton, Diana Z. O'Brien, Jennifer M. Piscopo

**Journal:** American Political Science Review (2026)

This notebook reproduces the analysis from the paper above.
It was auto-generated by [PoliRep](https://polirep.org).

---

## Setup

Install packages and import standard libraries.

In [ ]:
!pip install -q pandas numpy matplotlib scipy statsmodels pyfixest

import pandas as pd
import numpy as np
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

## Source Modules

Write the paper's `src/` package to disk so `from src.* import ...` works.

In [ ]:
import sys
from pathlib import Path

# Write src/ package to Colab filesystem
Path("src").mkdir(exist_ok=True)
Path("src/__init__.py").write_text("")
Path("src/config.py").write_text("\"\"\"Configuration: paths and constants for this paper's reproduction.\"\"\"\n\nfrom pathlib import Path\n\n# ---------------------------------------------------------------------------\n# Directory structure\n# ---------------------------------------------------------------------------\n\n# papers/<slug>/\nPAPER_ROOT = Path(\".\")\n\n# Top-level project root\nPROJECT_ROOT = PAPER_ROOT\n\n# Data paths\nDATA_RAW = Path(\"data/raw\")\nDATA_CONVERTED = Path(\"data/converted\")\n\n# Output paths\nOUTPUT_DIR = Path(\"output\")\nTABLE_DIR = OUTPUT_DIR / \"tables\"\nFIGURE_DIR = OUTPUT_DIR / \"figures\"\n\n# Original outputs for comparison\nORIGINAL_TABLE_DIR = Path(\"original/tables\")\nORIGINAL_FIGURE_DIR = Path(\"original/figures\")\n\n\n# ---------------------------------------------------------------------------\n# Analysis constants\n# ---------------------------------------------------------------------------\n\n# Treatment groups\nTREATMENTS = {\n    1: \"All-Male Panel (Sexual Harassment)\",\n    2: \"Gender-Balanced Panel (Sexual Harassment)\",\n    3: \"Quota-Gender-Balanced Panel (Sexual Harassment)\",\n    4: \"All-Male Panel (Animal Mistreatment)\",\n    5: \"Gender-Balanced Panel (Animal Mistreatment)\",\n    6: \"Quota-Gender-Balanced Panel (Animal Mistreatment)\",\n}\n\nHARASSMENT_TREATMENTS = [1, 2, 3]\nANIMAL_TREATMENTS = [4, 5, 6]\n\n# Outcome variables\nSUBSTANTIVE_LEG_VARS_HARASSMENT = [\"decide_citizen\", \"decide_women\", \"fair_women\"]\nSUBSTANTIVE_LEG_VARS_ANIMAL = [\"decide_citizen\", \"decide_animal\", \"fair_animal\"]\nPROCEDURAL_LEG_VARS = [\"decisionprocess\", \"council_trust\", \"overturnR\"]\n\n# Language-specific response coding mappings\nSPANISH_FAIRNESS_FEMININE = {\n    \"Muy injusta\": 1,\n    \"Algo injusta\": 2,\n    \"Algo justa\": 3,\n    \"Muy justa\": 4,\n}\n\nSPANISH_FAIRNESS_MASCULINE = {\n    \"Muy injusto\": 1,\n    \"Algo  injusto\": 2,\n    \"Algo injusto\": 2,\n    \"Algo justo\": 3,\n    \"Muy justo\": 4,\n}\n\nSPANISH_AGREEMENT = {\n    \"Muy en desacuerdo\": 1,\n    \"Algo en desacuerdo\": 2,\n    \"Algo de acuerdo\": 3,\n    \"Muy de acuerdo\": 4,\n}\n\nPORTUGUESE_FAIRNESS_FEMININE = {\n    \"Muito injusta\": 1,\n    \"Um pouco injusta\": 2,\n    \"Um pouco justa\": 3,\n    \"Um pouca justa\": 3,\n    \"Muito justa\": 4,\n}\n\nPORTUGUESE_FAIRNESS_MASCULINE = {\n    \"Muito injusto\": 1,\n    \"Um pouco injusto\": 2,\n    \"Um pouco justo\": 3,\n    \"Muito justo\": 4,\n}\n\nPORTUGUESE_AGREEMENT = {\n    \"Discordo totalmente\": 1,\n    \"Discordo\": 2,\n    \"Concordo\": 3,\n    \"Concordo totalmente\": 4,\n}\n\nENGLISH_FAIRNESS = {\n    \"Very unfair\": 1,\n    \"Unfair\": 2,\n    \"Somewhat fair\": 3,\n    \"Fair\": 3,\n    \"Very fair\": 4,\n}\n\nENGLISH_AGREEMENT = {\n    \"Strongly disagree\": 1,\n    \"Disagree\": 2,\n    \"Agree\": 3,\n    \"Strongly agree\": 4,\n    \"Strongly Agree\": 4,\n}\n\n# Gender coding\nGENDER_MAPPINGS = {\n    \"Mujer\": 1,\n    \"Woman\": 1,\n    \"Female\": 1,\n    \"Feminino\": 1,\n    \"Hombre\": 0,\n    \"Man\": 0,\n    \"Male\": 0,\n    \"Masculino\": 0,\n    \"Other\": None,\n    \"Otro/ No binario\": None,\n}\n\n# Ideology coding\nIDEOLOGY_MAPPINGS = {\n    \"Left\\n(0)\": 0,\n    \"Right\\n(10)\": 10,\n}\n\n# Country configurations\nCOUNTRY_CONFIGS = {\n    \"Argentina\": {\n        \"treat_var\": \"treat\",\n        \"fairness_map\": SPANISH_FAIRNESS_FEMININE | SPANISH_FAIRNESS_MASCULINE,\n        \"agreement_map\": SPANISH_AGREEMENT,\n        \"manip_check_harass\": \"Requerir capacitación en acoso sexual.\",\n        \"manip_check_animal\": \"Requerir capacitación en maltrato animal.\",\n        \"duration_filter\": None,\n        \"gender_var\": \"gender\",\n        \"ideology_var\": \"leftright_1\",\n    },\n    \"Brazil\": {\n        \"treat_var\": \"treat\",\n        \"fairness_map\": PORTUGUESE_FAIRNESS_FEMININE | PORTUGUESE_FAIRNESS_MASCULINE,\n        \"agreement_map\": PORTUGUESE_AGREEMENT,\n        \"manip_check_harass\": \"Exigir treinamento sobre assédio sexual.\",\n        \"manip_check_animal\": \"Exigir treinamento sobre maus-tratos a animais.\",\n        \"duration_filter\": None,\n        \"gender_var\": \"gender\",\n        \"ideology_var\": \"leftright_1\",\n    },\n    \"Mexico\": {\n        \"treat_var\": \"treat\",\n        \"fairness_map\": SPANISH_FAIRNESS_FEMININE | SPANISH_FAIRNESS_MASCULINE,\n        \"agreement_map\": SPANISH_AGREEMENT,\n        \"manip_check_harass\": None,  # Applied implicitly through subsetting\n        \"manip_check_animal\": None,\n        \"duration_filter\": 350,\n        \"gender_var\": \"gender\",\n        \"ideology_var\": \"leftright_1\",\n    },\n    \"Peru\": {\n        \"treat_var\": \"treat\",\n        \"fairness_map\": SPANISH_FAIRNESS_FEMININE | SPANISH_FAIRNESS_MASCULINE,\n        \"agreement_map\": SPANISH_AGREEMENT,\n        \"manip_check_harass\": None,\n        \"manip_check_animal\": None,\n        \"duration_filter\": 600,\n        \"gender_var\": \"gender\",\n        \"ideology_var\": \"leftright_1\",\n    },\n    \"NZ\": {\n        \"treat_var\": \"treat\",\n        \"fairness_map\": ENGLISH_FAIRNESS,\n        \"agreement_map\": ENGLISH_AGREEMENT,\n        \"manip_check_harass\": \"Require sexual harassment training\",\n        \"manip_check_animal\": \"Require animal mistreatment training\",\n        \"duration_filter\": None,\n        \"gender_var\": \"Q3.2\",\n        \"ideology_var\": \"Q27.1\",\n    },\n    \"USA\": {\n        \"treat_var\": \"treat\",\n        \"fairness_map\": ENGLISH_FAIRNESS,\n        \"agreement_map\": ENGLISH_AGREEMENT,\n        \"manip_check_harass\": \"Require sexual harassment training\",\n        \"manip_check_animal\": \"Require animal mistreatment training\",\n        \"duration_filter\": None,\n        \"gender_var\": \"gender\",\n        \"ideology_var\": \"leftright_1\",\n    },\n    \"UK\": {\n        \"treat_var\": \"Vignettes_DO\",\n        \"fairness_map\": ENGLISH_FAIRNESS,\n        \"agreement_map\": ENGLISH_AGREEMENT,\n        \"manip_check_harass\": None,\n        \"manip_check_animal\": None,\n        \"duration_filter\": 300,\n        \"gender_var\": \"gender\",\n        \"ideology_var\": \"leftright_1\",\n    },\n    \"Spain\": {\n        \"treat_var\": \"Vignettes_DO\",\n        \"fairness_map\": SPANISH_FAIRNESS_FEMININE | SPANISH_FAIRNESS_MASCULINE,\n        \"agreement_map\": SPANISH_AGREEMENT,\n        \"manip_check_harass\": None,\n        \"manip_check_animal\": None,\n        \"duration_filter\": 300,\n        \"gender_var\": \"gender\",\n        \"ideology_var\": \"leftright_1\",\n    },\n    \"Portugal\": {\n        \"treat_var\": \"Vignettes_DO\",\n        \"fairness_map\": PORTUGUESE_FAIRNESS_FEMININE | PORTUGUESE_FAIRNESS_MASCULINE,\n        \"agreement_map\": PORTUGUESE_AGREEMENT,\n        \"manip_check_harass\": None,\n        \"manip_check_animal\": None,\n        \"duration_filter\": 300,\n        \"gender_var\": \"gender\",\n        \"ideology_var\": \"leftright_1\",\n    },\n}\n\n\n# ---------------------------------------------------------------------------\n# Helpers\n# ---------------------------------------------------------------------------\n\ndef ensure_output_dirs():\n    \"\"\"Create output directories if they don't exist.\"\"\"\n    TABLE_DIR.mkdir(parents=True, exist_ok=True)\n    FIGURE_DIR.mkdir(parents=True, exist_ok=True)\n")
Path("src/country_processing.py").write_text("\"\"\"Process individual country datasets and create aggregated files.\n\nThis module replicates the individual country R scripts (Argentina.R, Brazil.R, etc.)\nthat process raw survey data, apply filters, create legitimacy indices, and export\naggregated datasets for merged analysis.\n\"\"\"\n\nimport pandas as pd\n\nfrom .config import (\n    ANIMAL_TREATMENTS,\n    COUNTRY_CONFIGS,\n    DATA_CONVERTED,\n    HARASSMENT_TREATMENTS,\n    PROCEDURAL_LEG_VARS,\n    SUBSTANTIVE_LEG_VARS_ANIMAL,\n    SUBSTANTIVE_LEG_VARS_HARASSMENT,\n)\nfrom .data import load_country_data\nfrom .helpers import omega_factor_analysis, recode_responses, reverse_code, standardize\n\n\ndef process_standard_country(country: str) -> pd.DataFrame:\n    \"\"\"Process a standard country dataset (Argentina, Brazil, Mexico, Peru, NZ, USA).\n\n    Args:\n        country: Country name\n\n    Returns:\n        Aggregated DataFrame ready for merging\n    \"\"\"\n    print(f\"  Processing {country}...\")\n\n    # Load raw data\n    df = load_country_data(country, use_agg=False)\n    config = COUNTRY_CONFIGS[country]\n\n    # Create treatment variable if needed\n    if \"treat\" not in df.columns and config[\"treat_var\"] == \"treat\":\n        # Construct from individual treatment indicators\n        treat_cols = [\n            \"Vignettes_DO_treat1\",\n            \"Vignettes_DO_treat2\",\n            \"Vignettes_DO_treat3\",\n            \"Vignettes_DO_treat4\",\n            \"Vignettes_DO_treat5\",\n            \"Vignettes_DO_treat6\",\n        ]\n\n        # Recode each treatment indicator\n        for i, col in enumerate(treat_cols, 1):\n            if col in df.columns:\n                df[col] = df[col].apply(lambda x: i if x == 1 else 0)\n\n        # Sum to create single treatment variable\n        df[\"treat\"] = sum(df[col] for col in treat_cols if col in df.columns)\n\n    # Apply duration filter if specified\n    if config[\"duration_filter\"] is not None:\n        duration_col = \"Duration (in seconds)\" if \"Duration (in seconds)\" in df.columns else \"Duration\"\n        if duration_col in df.columns:\n            df = df[df[duration_col] > config[\"duration_filter\"]].copy()\n\n    # Convert outcome variables to string for recoding\n    outcome_vars = [\n        \"decide_citizen\",\n        \"decide_women\",\n        \"decide_animal\",\n        \"fair_women\",\n        \"fair_animal\",\n        \"decisionprocess\",\n        \"overturn\",\n        \"council_trust\",\n        \"personalcouncil\",\n    ]\n\n    for var in outcome_vars:\n        if var in df.columns:\n            df[var] = df[var].astype(str)\n\n    # Recode responses using language-specific mappings\n    df = recode_responses(df, outcome_vars, config[\"fairness_map\"])\n    df = recode_responses(df, outcome_vars, config[\"agreement_map\"])\n\n    # Convert to numeric\n    for var in outcome_vars:\n        if var in df.columns:\n            df[var] = pd.to_numeric(df[var], errors=\"coerce\")\n\n    # Reverse code overturn\n    if \"overturn\" in df.columns:\n        df[\"overturnR\"] = reverse_code(df[\"overturn\"], min_val=1, max_val=4)\n\n    # Process harassment treatments\n    harass = df[df[\"treat\"].isin(HARASSMENT_TREATMENTS)].copy()\n\n    # Apply manipulation check for harassment\n    if config[\"manip_check_harass\"] is not None:\n        harass = harass[harass[\"decide_harass\"] == config[\"manip_check_harass\"]].copy()\n\n    # Create substantive legitimacy index for harassment\n    harass[\"SubLeg\"] = omega_factor_analysis(harass, SUBSTANTIVE_LEG_VARS_HARASSMENT, n_factors=1)\n    harass[\"SubLegStand\"] = standardize(harass[\"SubLeg\"])\n\n    # Create procedural legitimacy index for harassment\n    harass[\"ProLeg\"] = omega_factor_analysis(harass, PROCEDURAL_LEG_VARS, n_factors=1)\n    harass[\"ProLegStand\"] = standardize(harass[\"ProLeg\"])\n\n    # Process animal treatments\n    animal = df[df[\"treat\"].isin(ANIMAL_TREATMENTS)].copy()\n\n    # Apply manipulation check for animal\n    if config[\"manip_check_animal\"] is not None:\n        animal = animal[animal[\"decide_animal.1\"] == config[\"manip_check_animal\"]].copy()\n\n    # Create substantive legitimacy index for animal\n    animal[\"SubLeg\"] = omega_factor_analysis(animal, SUBSTANTIVE_LEG_VARS_ANIMAL, n_factors=1)\n    animal[\"SubLegStand\"] = standardize(animal[\"SubLeg\"])\n\n    # Create procedural legitimacy index for animal\n    animal[\"ProLeg\"] = omega_factor_analysis(animal, PROCEDURAL_LEG_VARS, n_factors=1)\n    animal[\"ProLegStand\"] = standardize(animal[\"ProLeg\"])\n\n    # Combine and select columns\n    keep_cols = [\"treat\", \"SubLegStand\", \"ProLegStand\", config[\"gender_var\"], config[\"ideology_var\"]]\n    harass_sub = harass[keep_cols].copy()\n    animal_sub = animal[keep_cols].copy()\n\n    # Rename columns to standard names\n    harass_sub = harass_sub.rename(\n        columns={config[\"gender_var\"]: \"gender\", config[\"ideology_var\"]: \"leftright_1\"}\n    )\n    animal_sub = animal_sub.rename(\n        columns={config[\"gender_var\"]: \"gender\", config[\"ideology_var\"]: \"leftright_1\"}\n    )\n\n    # Combine harassment and animal datasets\n    combined = pd.concat([harass_sub, animal_sub], ignore_index=True)\n    combined[\"Country\"] = country\n\n    return combined\n\n\ndef process_vignettes_do_country(country: str) -> pd.DataFrame:\n    \"\"\"Process countries with Vignettes_DO treatment variable (UK, Spain, Portugal).\n\n    Args:\n        country: Country name\n\n    Returns:\n        Aggregated DataFrame ready for merging\n    \"\"\"\n    print(f\"  Processing {country}...\")\n\n    # Load raw data\n    df = load_country_data(country, use_agg=False)\n    config = COUNTRY_CONFIGS[country]\n\n    # Apply duration filter\n    if config[\"duration_filter\"] is not None:\n        duration_col = \"Duration (in seconds)\" if \"Duration (in seconds)\" in df.columns else \"Duration\"\n        if duration_col in df.columns:\n            df = df[df[duration_col] > config[\"duration_filter\"]].copy()\n\n    # Convert outcome variables to string\n    outcome_vars = [\n        \"decide_citizen\",\n        \"decide_women\",\n        \"decide_animal\",\n        \"fair_women\",\n        \"fair_animal\",\n        \"decisionprocess\",\n        \"overturn\",\n        \"council_trust\",\n        \"personalcouncil\",\n    ]\n\n    for var in outcome_vars:\n        if var in df.columns:\n            df[var] = df[var].astype(str)\n\n    # Recode responses\n    df = recode_responses(df, outcome_vars, config[\"fairness_map\"])\n    df = recode_responses(df, outcome_vars, config[\"agreement_map\"])\n\n    # Convert to numeric\n    for var in outcome_vars:\n        if var in df.columns:\n            df[var] = pd.to_numeric(df[var], errors=\"coerce\")\n\n    # Reverse code overturn\n    if \"overturn\" in df.columns:\n        df[\"overturnR\"] = reverse_code(df[\"overturn\"], min_val=1, max_val=4)\n\n    # Create numeric treatment variable from Vignettes_DO\n    treat_mapping = {\n        \"treat1\": 1,\n        \"treat2\": 2,\n        \"treat3\": 3,\n        \"treat4\": 4,\n        \"treat5\": 5,\n        \"treat6\": 6,\n    }\n    df[\"treat\"] = df[\"Vignettes_DO\"].replace(treat_mapping)\n\n    # Process harassment treatments\n    harass = df[df[\"treat\"].isin(HARASSMENT_TREATMENTS)].copy()\n    harass[\"SubLeg\"] = omega_factor_analysis(harass, SUBSTANTIVE_LEG_VARS_HARASSMENT, n_factors=1)\n    harass[\"SubLegStand\"] = standardize(harass[\"SubLeg\"])\n    harass[\"ProLeg\"] = omega_factor_analysis(harass, PROCEDURAL_LEG_VARS, n_factors=1)\n    harass[\"ProLegStand\"] = standardize(harass[\"ProLeg\"])\n\n    # Process animal treatments\n    animal = df[df[\"treat\"].isin(ANIMAL_TREATMENTS)].copy()\n    animal[\"SubLeg\"] = omega_factor_analysis(animal, SUBSTANTIVE_LEG_VARS_ANIMAL, n_factors=1)\n    animal[\"SubLegStand\"] = standardize(animal[\"SubLeg\"])\n    animal[\"ProLeg\"] = omega_factor_analysis(animal, PROCEDURAL_LEG_VARS, n_factors=1)\n    animal[\"ProLegStand\"] = standardize(animal[\"ProLeg\"])\n\n    # Combine and select columns\n    keep_cols = [\"treat\", \"SubLegStand\", \"ProLegStand\", \"gender\", \"leftright_1\"]\n    harass_sub = harass[keep_cols].copy()\n    animal_sub = animal[keep_cols].copy()\n\n    combined = pd.concat([harass_sub, animal_sub], ignore_index=True)\n    combined[\"Country\"] = country\n\n    return combined\n\n\ndef run():\n    \"\"\"Run country processing for all countries and save aggregated datasets.\"\"\"\n    print(\"Processing country datasets...\")\n\n    # Process standard countries\n    standard_countries = [\"Argentina\", \"Brazil\", \"Mexico\", \"Peru\", \"NZ\", \"USA\"]\n    for country in standard_countries:\n        try:\n            agg_df = process_standard_country(country)\n            # Save aggregated dataset\n            output_file = DATA_CONVERTED / f\"{country.replace(' ', '')}Agg_python.csv\"\n            agg_df.to_csv(output_file, index=False)\n            print(f\"    Saved {output_file.name} ({len(agg_df)} rows)\")\n        except Exception as e:\n            print(f\"    ERROR processing {country}: {e}\")\n\n    # Process Vignettes_DO countries\n    vignettes_countries = [\"UK\", \"Spain\", \"Portugal\"]\n    for country in vignettes_countries:\n        try:\n            agg_df = process_vignettes_do_country(country)\n            output_file = DATA_CONVERTED / f\"{country}Agg_python.csv\"\n            agg_df.to_csv(output_file, index=False)\n            print(f\"    Saved {output_file.name} ({len(agg_df)} rows)\")\n        except Exception as e:\n            print(f\"    ERROR processing {country}: {e}\")\n\n    print(\"Country processing complete!\")\n")
Path("src/data.py").write_text("\"\"\"Data loading and sample construction.\n\nPortal compatibility requires these three functions:\n  - load_main_data() -> pd.DataFrame\n  - prepare_analysis_sample(df) -> pd.DataFrame\n  - get_sample_stats(sample) -> dict\n\"\"\"\n\nimport pandas as pd\n\ntry:\n    from .config import DATA_CONVERTED, GENDER_MAPPINGS, IDEOLOGY_MAPPINGS\nexcept ImportError:\n    # Fallback for direct script execution\n    from config import DATA_CONVERTED, GENDER_MAPPINGS, IDEOLOGY_MAPPINGS\n\n\ndef load_country_data(country: str, use_agg: bool = False) -> pd.DataFrame:\n    \"\"\"Load data for a specific country.\n\n    Args:\n        country: Country name (e.g., \"Argentina\", \"Brazil\", \"Mexico\")\n        use_agg: If True, load the aggregated file (e.g., ArgAgg.parquet)\n\n    Returns:\n        DataFrame with country data\n    \"\"\"\n    if use_agg:\n        if country == \"NorAusFrance\":\n            filename = \"NorAusFranceAgg.parquet\"\n        else:\n            # Map country names to aggregated filenames\n            agg_names = {\n                \"Argentina\": \"ArgAgg\",\n                \"Brazil\": \"BrazilAgg\",\n                \"Mexico\": \"MexAgg\",\n                \"Peru\": \"PeruAgg\",\n                \"NZ\": \"NZAgg\",\n                \"USA\": \"USAgg\",\n                \"UK\": \"UKAgg\",\n                \"Spain\": \"SpainAgg\",\n                \"Portugal\": \"PortAgg\",\n            }\n            filename = f\"{agg_names[country]}.parquet\"\n    else:\n        # Load complete country files\n        if country == \"NorAusFrance\":\n            filename = \"AusNorFrance.parquet\"\n        elif country in [\"UK\", \"Spain\", \"Portugal\"]:\n            filename = f\"{country}Complete.parquet\"\n        else:\n            filename = f\"{country}.parquet\"\n\n    filepath = DATA_CONVERTED / filename\n    return pd.read_parquet(filepath)\n\n\ndef load_main_data() -> pd.DataFrame:\n    \"\"\"Load the merged analysis dataset from all country aggregated files.\n\n    Returns:\n        DataFrame with all countries merged together.\n    \"\"\"\n    # Load all aggregated country datasets\n    countries = [\n        \"Mexico\",\n        \"Peru\",\n        \"UK\",\n        \"Spain\",\n        \"Portugal\",\n        \"USA\",\n        \"Brazil\",\n        \"Argentina\",\n        \"NZ\",\n        \"NorAusFrance\",\n    ]\n\n    dfs = []\n    for country in countries:\n        df = load_country_data(country, use_agg=True)\n\n        # Rename Vignettes_DO to treat for UK, Spain, Portugal\n        if \"Vignettes_DO\" in df.columns and \"treat\" not in df.columns:\n            df = df.rename(columns={\"Vignettes_DO\": \"treat\"})\n\n        # Rename QCOUNTRY to Country for NorAusFrance\n        if \"QCOUNTRY\" in df.columns and \"Country\" not in df.columns:\n            df = df.rename(columns={\"QCOUNTRY\": \"Country\"})\n\n        dfs.append(df)\n\n    # Merge all datasets\n    merged = pd.concat(dfs, ignore_index=True)\n\n    return merged\n\n\ndef prepare_analysis_sample(df: pd.DataFrame) -> pd.DataFrame:\n    \"\"\"Apply sample restrictions and construct derived variables.\n\n    Args:\n        df: Raw loaded DataFrame from load_main_data().\n\n    Returns:\n        Analysis-ready DataFrame with all filters and transformations applied.\n    \"\"\"\n    df = df.copy()\n\n    # Recode treatment variable (convert text to numeric if needed)\n    treat_mapping = {\n        \"treat1\": 1,\n        \"treat2\": 2,\n        \"treat3\": 3,\n        \"treat4\": 4,\n        \"treat5\": 5,\n        \"treat6\": 6,\n    }\n    df[\"treat\"] = df[\"treat\"].replace(treat_mapping)\n    df[\"treat\"] = pd.to_numeric(df[\"treat\"], errors=\"coerce\")\n\n    # Recode gender\n    df[\"gender\"] = df[\"gender\"].replace(GENDER_MAPPINGS)\n    df[\"gender\"] = pd.to_numeric(df[\"gender\"], errors=\"coerce\")\n\n    # Recode ideology\n    df[\"leftright_1\"] = df[\"leftright_1\"].replace(IDEOLOGY_MAPPINGS)\n    df[\"leftright_1\"] = pd.to_numeric(df[\"leftright_1\"], errors=\"coerce\")\n\n    return df\n\n\ndef get_sample_stats(sample: pd.DataFrame) -> dict:\n    \"\"\"Compute summary statistics for the analysis sample.\n\n    Args:\n        sample: Prepared analysis sample from prepare_analysis_sample().\n\n    Returns:\n        Dictionary with keys like n_obs, n_countries, treatment_distribution, etc.\n    \"\"\"\n    stats = {\n        \"n_obs\": len(sample),\n        \"n_countries\": sample[\"Country\"].nunique() if \"Country\" in sample.columns else None,\n        \"n_complete_subleg\": sample[\"SubLegStand\"].notna().sum()\n        if \"SubLegStand\" in sample.columns\n        else None,\n        \"n_complete_proleg\": sample[\"ProLegStand\"].notna().sum()\n        if \"ProLegStand\" in sample.columns\n        else None,\n    }\n\n    if \"treat\" in sample.columns:\n        stats[\"treatment_distribution\"] = sample[\"treat\"].value_counts().to_dict()\n\n    if \"Country\" in sample.columns:\n        stats[\"country_distribution\"] = sample[\"Country\"].value_counts().to_dict()\n\n    return stats\n")
Path("src/helpers.py").write_text("\"\"\"Helper functions for factor analysis, standardization, and summary statistics.\"\"\"\n\nimport numpy as np\nimport pandas as pd\nfrom scipy import stats\n\n\ndef omega_factor_analysis(df: pd.DataFrame, variables: list[str], n_factors: int = 1) -> np.ndarray:\n    \"\"\"Perform single-factor extraction via principal axis factoring.\n\n    Approximates R's psych::omega() with nfactors=1 using eigendecomposition\n    of the correlation matrix and the Thurstone regression method for scores.\n\n    Args:\n        df: DataFrame containing the variables\n        variables: List of variable names to include in factor analysis\n        n_factors: Number of factors to extract (only 1 supported)\n\n    Returns:\n        Array of factor scores for each observation\n    \"\"\"\n    # Extract complete cases only\n    data = df[variables].copy()\n    data_clean = data.dropna()\n\n    if len(data_clean) == 0:\n        raise ValueError(f\"No complete cases for variables: {variables}\")\n\n    # Standardize to z-scores\n    means = data_clean.mean()\n    stds = data_clean.std(ddof=1)\n    z = (data_clean - means) / stds\n\n    # Correlation matrix\n    R = z.corr().values\n\n    # Eigendecomposition\n    eigenvalues, eigenvectors = np.linalg.eigh(R)\n    # Sort descending\n    idx = np.argsort(eigenvalues)[::-1]\n    eigenvalues = eigenvalues[idx]\n    eigenvectors = eigenvectors[:, idx]\n\n    # First factor loadings\n    loadings = eigenvectors[:, 0] * np.sqrt(max(eigenvalues[0], 0))\n\n    # Factor scores via Thurstone regression method: scores = Z @ R_inv @ loadings\n    R_inv = np.linalg.pinv(R)\n    weights = R_inv @ loadings\n    scores = z.values @ weights\n\n    # Map back to full DataFrame with NaN for missing cases\n    result = np.full(len(df), np.nan)\n    for i, original_idx in enumerate(data_clean.index):\n        pos = df.index.get_loc(original_idx)\n        result[pos] = scores[i]\n\n    return result\n\n\ndef standardize(x: pd.Series | np.ndarray) -> pd.Series | np.ndarray:\n    \"\"\"Standardize a variable to mean=0, sd=1.\n\n    Matches R's effectsize::standardize() function.\n\n    Args:\n        x: Variable to standardize\n\n    Returns:\n        Standardized variable with same type as input\n    \"\"\"\n    if isinstance(x, pd.Series):\n        mean = x.mean()\n        std = x.std(ddof=1)  # Use sample std like R\n        return (x - mean) / std\n    else:\n        mean = np.nanmean(x)\n        std = np.nanstd(x, ddof=1)\n        return (x - mean) / std\n\n\ndef summary_se(\n    data: pd.DataFrame,\n    measurevar: str,\n    groupvars: list[str] | None = None,\n    na_rm: bool = False,\n    conf_interval: float = 0.95,\n) -> pd.DataFrame:\n    \"\"\"Calculate summary statistics with standard errors and confidence intervals.\n\n    Replicates the summarySE function from the R script.\n\n    Args:\n        data: DataFrame containing the data\n        measurevar: Name of the variable to summarize\n        groupvars: List of grouping variables (optional)\n        na_rm: Whether to remove NA values\n        conf_interval: Confidence interval level (default 0.95)\n\n    Returns:\n        DataFrame with N, mean, sd, se, and ci for each group\n    \"\"\"\n    if groupvars is None:\n        groupvars = []\n\n    def length2(x, na_rm=False):\n        if na_rm:\n            return (~pd.isna(x)).sum()\n        else:\n            return len(x)\n\n    def summarize(group):\n        col = group[measurevar]\n        n = length2(col, na_rm=na_rm)\n        mean_val = col.mean() if na_rm else col.mean()\n        sd_val = col.std(ddof=1) if na_rm else col.std(ddof=1)\n        return pd.Series({\"N\": n, measurevar: mean_val, \"sd\": sd_val})\n\n    if groupvars:\n        # Group and apply summarize function\n        grouped = data.groupby(groupvars, as_index=False)\n        result_list = []\n        for name, group in grouped:\n            row_data = summarize(group).to_dict()\n            if isinstance(name, tuple):\n                for i, gvar in enumerate(groupvars):\n                    row_data[gvar] = name[i]\n            else:\n                row_data[groupvars[0]] = name\n            result_list.append(row_data)\n        result = pd.DataFrame(result_list)\n    else:\n        result = pd.DataFrame([summarize(data)])\n\n    # Calculate standard error\n    result[\"se\"] = result[\"sd\"] / np.sqrt(result[\"N\"])\n\n    # Calculate confidence interval\n    # Using t-distribution with df = N - 1\n    ci_mult = stats.t.ppf((conf_interval / 2 + 0.5), result[\"N\"] - 1)\n    result[\"ci\"] = result[\"se\"] * ci_mult\n\n    return result\n\n\ndef recode_responses(\n    df: pd.DataFrame, variables: list[str], mapping: dict\n) -> pd.DataFrame:\n    \"\"\"Recode text responses to numeric values.\n\n    Args:\n        df: DataFrame to recode\n        variables: List of variable names to recode\n        mapping: Dictionary mapping text values to numeric codes\n\n    Returns:\n        DataFrame with recoded variables\n    \"\"\"\n    df = df.copy()\n    for var in variables:\n        if var in df.columns:\n            df[var] = df[var].replace(mapping)\n    return df\n\n\ndef reverse_code(x: pd.Series | np.ndarray, min_val: int = 1, max_val: int = 4) -> pd.Series | np.ndarray:\n    \"\"\"Reverse code a variable (e.g., 1->4, 2->3, 3->2, 4->1).\n\n    Args:\n        x: Variable to reverse code\n        min_val: Minimum value in scale\n        max_val: Maximum value in scale\n\n    Returns:\n        Reverse coded variable\n    \"\"\"\n    if isinstance(x, pd.Series):\n        return (max_val + min_val) - x\n    else:\n        return (max_val + min_val) - x\n")
Path("src/merged_analysis.py").write_text("\"\"\"Merged cross-country analysis and figure generation.\n\nThis module replicates MergedAnalysis.R, which:\n1. Loads all country aggregated datasets\n2. Standardizes variables across countries\n3. Creates Figure 2 (main treatment effects)\n4. Creates Figure 4 (USA vs UK comparison)\n\"\"\"\n\nimport matplotlib.pyplot as plt\nimport numpy as np\nimport pandas as pd\n\nfrom .config import FIGURE_DIR, HARASSMENT_TREATMENTS\nfrom .data import load_main_data, prepare_analysis_sample\nfrom .helpers import summary_se\n\n\ndef create_figure_2(merged_df: pd.DataFrame):\n    \"\"\"Create Figure 2: Main treatment effects on substantive and procedural legitimacy.\n\n    Args:\n        merged_df: Merged dataset with all countries\n    \"\"\"\n    # Subset to harassment treatments\n    harass = merged_df[merged_df[\"treat\"].isin(HARASSMENT_TREATMENTS)].copy()\n\n    # Remove NA values from substantive legitimacy\n    harass_sub = harass[harass[\"SubLegStand\"].notna()].copy()\n\n    # Calculate summary statistics for substantive legitimacy\n    data_sum1 = summary_se(harass_sub, measurevar=\"SubLegStand\", groupvars=[\"treat\"])\n\n    # Remove NA values from procedural legitimacy\n    harass_pro = harass[harass[\"ProLegStand\"].notna()].copy()\n\n    # Calculate summary statistics for procedural legitimacy\n    data_sum2 = summary_se(harass_pro, measurevar=\"ProLegStand\", groupvars=[\"treat\"])\n\n    # Create figure with 2 subplots\n    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 5))\n\n    # Plot 1: Substantive Legitimacy\n    x_pos = np.array([0.97, 1.0, 1.03])  # Offset x positions slightly\n    ax1.errorbar(\n        x_pos,\n        data_sum1[\"SubLegStand\"],\n        yerr=data_sum1[\"se\"],\n        fmt=\"o\",\n        color=\"black\",\n        markersize=8,\n        markerfacecolor=\"white\",\n        capsize=0,\n        linewidth=1.5,\n    )\n    ax1.axhline(y=0, color=\"gray\", linestyle=\"--\", alpha=0.5)\n    ax1.set_xlim(0.95, 1.05)\n    ax1.set_ylim(-1, 1)\n    ax1.set_ylabel(\"Average Substantive Legitimacy\\n(standardized within countries)\", fontsize=11)\n    ax1.set_title(\n        \"Women's rights issue (sexual harassment)\\nSubstantive legitimacy beliefs\", fontsize=11\n    )\n    ax1.set_xticks([])\n\n    # Add treatment labels\n    ax1.text(\n        0.9675,\n        -0.05,\n        \"All-Male\\nCouncil\",\n        ha=\"center\",\n        va=\"top\",\n        fontsize=9,\n        transform=ax1.get_xaxis_transform(),\n    )\n    ax1.text(\n        1.0,\n        0.28,\n        \"Gender-Balanced\\nCouncil\",\n        ha=\"center\",\n        va=\"bottom\",\n        fontsize=9,\n        transform=ax1.get_xaxis_transform(),\n    )\n    ax1.text(\n        1.032,\n        0.25,\n        \"Quota-Elected\\nGender-Balanced\\nCouncil\",\n        ha=\"center\",\n        va=\"bottom\",\n        fontsize=9,\n        transform=ax1.get_xaxis_transform(),\n    )\n\n    # Plot 2: Procedural Legitimacy\n    ax2.errorbar(\n        x_pos,\n        data_sum2[\"ProLegStand\"],\n        yerr=data_sum2[\"se\"],\n        fmt=\"o\",\n        color=\"black\",\n        markersize=8,\n        markerfacecolor=\"white\",\n        capsize=0,\n        linewidth=1.5,\n    )\n    ax2.axhline(y=0, color=\"gray\", linestyle=\"--\", alpha=0.5)\n    ax2.set_xlim(0.95, 1.05)\n    ax2.set_ylim(-1, 1)\n    ax2.set_ylabel(\"Average Procedural Legitimacy\\n(standardized within countries)\", fontsize=11)\n    ax2.set_title(\n        \"Women's rights issue (sexual harassment)\\nProcedural legitimacy beliefs\", fontsize=11\n    )\n    ax2.set_xticks([])\n\n    # Add treatment labels\n    ax2.text(\n        0.9675,\n        -0.42,\n        \"All-Male\\nCouncil\",\n        ha=\"center\",\n        va=\"top\",\n        fontsize=9,\n        transform=ax2.get_xaxis_transform(),\n    )\n    ax2.text(\n        1.0,\n        0.45,\n        \"Gender-Balanced\\nCouncil\",\n        ha=\"center\",\n        va=\"bottom\",\n        fontsize=9,\n        transform=ax2.get_xaxis_transform(),\n    )\n    ax2.text(\n        1.032,\n        0.38,\n        \"Quota-Elected\\nGender-Balanced\\nCouncil\",\n        ha=\"center\",\n        va=\"bottom\",\n        fontsize=9,\n        transform=ax2.get_xaxis_transform(),\n    )\n\n    plt.tight_layout()\n\n    # Save figure\n    output_file = FIGURE_DIR / \"Figure2.pdf\"\n    plt.savefig(output_file, bbox_inches=\"tight\")\n    print(f\"  Saved {output_file}\")\n\n    # Also save as PNG\n    output_png = FIGURE_DIR / \"Figure2.png\"\n    plt.savefig(output_png, bbox_inches=\"tight\", dpi=300)\n    print(f\"  Saved {output_png}\")\n\n    plt.close()\n\n\ndef create_figure_4(merged_df: pd.DataFrame):\n    \"\"\"Create Figure 4: USA vs UK comparison.\n\n    Args:\n        merged_df: Merged dataset with all countries\n    \"\"\"\n    # Subset to harassment treatments\n    harass = merged_df[merged_df[\"treat\"].isin(HARASSMENT_TREATMENTS)].copy()\n\n    # Filter for USA and UK\n    usa = harass[harass[\"Country\"] == \"USA\"].copy()\n    uk = harass[harass[\"Country\"] == \"UK\"].copy()\n\n    # Create figure with 4 subplots (2x2)\n    fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(10, 8))\n\n    # USA Substantive Legitimacy\n    usa_sub = usa[usa[\"SubLegStand\"].notna()].copy()\n    data_sum1 = summary_se(usa_sub, measurevar=\"SubLegStand\", groupvars=[\"treat\"])\n\n    x_pos = np.array([0.97, 1.0, 1.03])\n    ax1.errorbar(\n        x_pos,\n        data_sum1[\"SubLegStand\"],\n        yerr=data_sum1[\"se\"],\n        fmt=\"o\",\n        color=\"black\",\n        markersize=6,\n        markerfacecolor=\"white\",\n        capsize=0,\n        linewidth=1.5,\n    )\n    ax1.axhline(y=0, color=\"gray\", linestyle=\"--\", alpha=0.5)\n    ax1.set_xlim(0.95, 1.05)\n    ax1.set_ylim(-1, 1)\n    ax1.set_ylabel(\"Average Substantive Legitimacy\\n(standardized within countries)\", fontsize=10)\n    ax1.set_title(\n        \"Women's rights issue (sexual harassment)\\nSubstantive legitimacy beliefs\\nUS sample\",\n        fontsize=10,\n    )\n    ax1.set_xticks([])\n\n    # USA Procedural Legitimacy\n    usa_pro = usa[usa[\"ProLegStand\"].notna()].copy()\n    data_sum2 = summary_se(usa_pro, measurevar=\"ProLegStand\", groupvars=[\"treat\"])\n\n    ax2.errorbar(\n        x_pos,\n        data_sum2[\"ProLegStand\"],\n        yerr=data_sum2[\"se\"],\n        fmt=\"o\",\n        color=\"black\",\n        markersize=6,\n        markerfacecolor=\"white\",\n        capsize=0,\n        linewidth=1.5,\n    )\n    ax2.axhline(y=0, color=\"gray\", linestyle=\"--\", alpha=0.5)\n    ax2.set_xlim(0.95, 1.05)\n    ax2.set_ylim(-1, 1)\n    ax2.set_ylabel(\"Average Procedural Legitimacy\\n(standardized within countries)\", fontsize=10)\n    ax2.set_title(\n        \"Women's rights issue (sexual harassment)\\nProcedural legitimacy beliefs\\nUS sample\",\n        fontsize=10,\n    )\n    ax2.set_xticks([])\n\n    # UK Substantive Legitimacy\n    uk_sub = uk[uk[\"SubLegStand\"].notna()].copy()\n    data_sum3 = summary_se(uk_sub, measurevar=\"SubLegStand\", groupvars=[\"treat\"])\n\n    ax3.errorbar(\n        x_pos,\n        data_sum3[\"SubLegStand\"],\n        yerr=data_sum3[\"se\"],\n        fmt=\"o\",\n        color=\"black\",\n        markersize=6,\n        markerfacecolor=\"white\",\n        capsize=0,\n        linewidth=1.5,\n    )\n    ax3.axhline(y=0, color=\"gray\", linestyle=\"--\", alpha=0.5)\n    ax3.set_xlim(0.95, 1.05)\n    ax3.set_ylim(-1, 1)\n    ax3.set_ylabel(\"Average Substantive Legitimacy\\n(standardized within countries)\", fontsize=10)\n    ax3.set_title(\n        \"Women's rights issue (sexual harassment)\\nSubstantive legitimacy beliefs\\nUK sample\",\n        fontsize=10,\n    )\n    ax3.set_xticks([])\n\n    # UK Procedural Legitimacy\n    uk_pro = uk[uk[\"ProLegStand\"].notna()].copy()\n    data_sum4 = summary_se(uk_pro, measurevar=\"ProLegStand\", groupvars=[\"treat\"])\n\n    ax4.errorbar(\n        x_pos,\n        data_sum4[\"ProLegStand\"],\n        yerr=data_sum4[\"se\"],\n        fmt=\"o\",\n        color=\"black\",\n        markersize=6,\n        markerfacecolor=\"white\",\n        capsize=0,\n        linewidth=1.5,\n    )\n    ax4.axhline(y=0, color=\"gray\", linestyle=\"--\", alpha=0.5)\n    ax4.set_xlim(0.95, 1.05)\n    ax4.set_ylim(-1, 1)\n    ax4.set_ylabel(\"Average Procedural Legitimacy\\n(standardized within countries)\", fontsize=10)\n    ax4.set_title(\n        \"Women's rights issue (sexual harassment)\\nProcedural legitimacy beliefs\\nUK sample\",\n        fontsize=10,\n    )\n    ax4.set_xticks([])\n\n    plt.tight_layout()\n\n    # Save figure\n    output_file = FIGURE_DIR / \"Figure4.pdf\"\n    plt.savefig(output_file, bbox_inches=\"tight\")\n    print(f\"  Saved {output_file}\")\n\n    output_png = FIGURE_DIR / \"Figure4.png\"\n    plt.savefig(output_png, bbox_inches=\"tight\", dpi=300)\n    print(f\"  Saved {output_png}\")\n\n    plt.close()\n\n\ndef run():\n    \"\"\"Run merged analysis and generate figures.\"\"\"\n    print(\"Running merged analysis...\")\n\n    # Load merged data\n    merged = load_main_data()\n    print(f\"  Loaded {len(merged)} observations from {merged['Country'].nunique()} countries\")\n\n    # Prepare analysis sample\n    merged = prepare_analysis_sample(merged)\n\n    # Create figures\n    print(\"  Creating Figure 2...\")\n    create_figure_2(merged)\n\n    print(\"  Creating Figure 4...\")\n    create_figure_4(merged)\n\n    print(\"Merged analysis complete!\")\n")

# Ensure src is importable
sys.path.insert(0, ".")

# Create output directories
Path("output/tables").mkdir(parents=True, exist_ok=True)
Path("output/figures").mkdir(parents=True, exist_ok=True)

## Data Preparation


### Load Country-Level Aggregated Data

Load all 10 country-level aggregated datasets (ArgAgg, BrazilAgg,
MexAgg, PeruAgg, NZAgg, USAgg, UKAgg, SpainAgg, PortAgg, NorAusFranceAgg).
Each file contains pre-computed legitimacy indices and cleaned demographic
variables.

In [ ]:
def load_main_data() -> pd.DataFrame:
    """Load the merged analysis dataset from all country aggregated files.

    Returns:
        DataFrame with all countries merged together.
    """
    # Load all aggregated country datasets
    countries = [
        "Mexico",
        "Peru",
        "UK",
        "Spain",
        "Portugal",
        "USA",
        "Brazil",
        "Argentina",
        "NZ",
        "NorAusFrance",
    ]

    dfs = []
    for country in countries:
        df = load_country_data(country, use_agg=True)

        # Rename Vignettes_DO to treat for UK, Spain, Portugal
        if "Vignettes_DO" in df.columns and "treat" not in df.columns:
            df = df.rename(columns={"Vignettes_DO": "treat"})

        # Rename QCOUNTRY to Country for NorAusFrance
        if "QCOUNTRY" in df.columns and "Country" not in df.columns:
            df = df.rename(columns={"QCOUNTRY": "Country"})

        dfs.append(df)

    # Merge all datasets
    merged = pd.concat(dfs, ignore_index=True)

    return merged

In [ ]:
df = load_main_data()
print(f"Shape: {df.shape}")
print(f"Countries:")
print(df['Country'].value_counts())
print(f"\nTreatment distribution (raw, mixed types):")
print(df['treat'].value_counts())
display(df.head())


### Recode Treatment Variable

Convert text treatment labels to numeric codes. UK/Spain/Portugal use
"treat1"..."treat6" labels; other countries use numeric 1-6. Standardize
to numeric 1-6 for all countries.

In [ ]:
def prepare_analysis_sample(df: pd.DataFrame) -> pd.DataFrame:
    """Apply sample restrictions and construct derived variables.

    Args:
        df: Raw loaded DataFrame from load_main_data().

    Returns:
        Analysis-ready DataFrame with all filters and transformations applied.
    """
    df = df.copy()

    # Recode treatment variable (convert text to numeric if needed)
    treat_mapping = {
        "treat1": 1,
        "treat2": 2,
        "treat3": 3,
        "treat4": 4,
        "treat5": 5,
        "treat6": 6,
    }
    df["treat"] = df["treat"].replace(treat_mapping)
    df["treat"] = pd.to_numeric(df["treat"], errors="coerce")

    # Recode gender
    df["gender"] = df["gender"].replace(GENDER_MAPPINGS)
    df["gender"] = pd.to_numeric(df["gender"], errors="coerce")

    # Recode ideology
    df["leftright_1"] = df["leftright_1"].replace(IDEOLOGY_MAPPINGS)
    df["leftright_1"] = pd.to_numeric(df["leftright_1"], errors="coerce")

    return df

In [ ]:
sample = prepare_analysis_sample(load_main_data())
print(f"Treatment coding (after recode):")
print(sample['treat'].value_counts().sort_index())
print(f"\nTreatment type: {sample['treat'].dtype}")
print(f"Missing treatments: {sample['treat'].isna().sum()}")


### Recode Gender and Ideology

Recode gender from language-specific text labels to binary (1=Female, 0=Male).
Recode ideology from text labels to numeric 0-10 scale. Apply country-specific
mappings from config.

In [ ]:
def prepare_analysis_sample(df: pd.DataFrame) -> pd.DataFrame:
    """Apply sample restrictions and construct derived variables.

    Args:
        df: Raw loaded DataFrame from load_main_data().

    Returns:
        Analysis-ready DataFrame with all filters and transformations applied.
    """
    df = df.copy()

    # Recode treatment variable (convert text to numeric if needed)
    treat_mapping = {
        "treat1": 1,
        "treat2": 2,
        "treat3": 3,
        "treat4": 4,
        "treat5": 5,
        "treat6": 6,
    }
    df["treat"] = df["treat"].replace(treat_mapping)
    df["treat"] = pd.to_numeric(df["treat"], errors="coerce")

    # Recode gender
    df["gender"] = df["gender"].replace(GENDER_MAPPINGS)
    df["gender"] = pd.to_numeric(df["gender"], errors="coerce")

    # Recode ideology
    df["leftright_1"] = df["leftright_1"].replace(IDEOLOGY_MAPPINGS)
    df["leftright_1"] = pd.to_numeric(df["leftright_1"], errors="coerce")

    return df

In [ ]:
sample = prepare_analysis_sample(load_main_data())
print(f"Gender distribution:")
print(sample['gender'].value_counts())
print(f"\nIdeology summary:")
print(sample['leftright_1'].describe())
print(f"\nIdeology missing: {sample['leftright_1'].isna().sum()} ({sample['leftright_1'].isna().mean():.1%})")


### Compute Sample Statistics

Calculate sample size, country distribution, treatment balance, and
legitimacy outcome completeness. Report missing data patterns.

In [ ]:
def get_sample_stats(sample: pd.DataFrame) -> dict:
    """Compute summary statistics for the analysis sample.

    Args:
        sample: Prepared analysis sample from prepare_analysis_sample().

    Returns:
        Dictionary with keys like n_obs, n_countries, treatment_distribution, etc.
    """
    stats = {
        "n_obs": len(sample),
        "n_countries": sample["Country"].nunique() if "Country" in sample.columns else None,
        "n_complete_subleg": sample["SubLegStand"].notna().sum()
        if "SubLegStand" in sample.columns
        else None,
        "n_complete_proleg": sample["ProLegStand"].notna().sum()
        if "ProLegStand" in sample.columns
        else None,
    }

    if "treat" in sample.columns:
        stats["treatment_distribution"] = sample["treat"].value_counts().to_dict()

    if "Country" in sample.columns:
        stats["country_distribution"] = sample["Country"].value_counts().to_dict()

    return stats

In [ ]:
sample = prepare_analysis_sample(load_main_data())
stats = get_sample_stats(sample)
print(f"Total observations: {stats['n_obs']}")
print(f"Countries: {stats['n_countries']}")
print(f"Complete SubLegStand: {stats['n_complete_subleg']}")
print(f"Complete ProLegStand: {stats['n_complete_proleg']}")
print(f"\nCountry distribution:")
for country, count in sorted(stats['country_distribution'].items()):
    print(f"  {country}: {count}")
print(f"\nTreatment distribution:")
for treat, count in sorted(stats['treatment_distribution'].items()):
    print(f"  Treatment {treat}: {count}")


### Create Treatment Subsets

Split the merged dataset into sexual harassment vignettes (treatments 1-3)
and animal mistreatment vignettes (treatments 4-6) for separate analysis.
Australia/Norway/France only have harassment vignettes.

In [ ]:
def prepare_analysis_sample(df: pd.DataFrame) -> pd.DataFrame:
    """Apply sample restrictions and construct derived variables.

    Args:
        df: Raw loaded DataFrame from load_main_data().

    Returns:
        Analysis-ready DataFrame with all filters and transformations applied.
    """
    df = df.copy()

    # Recode treatment variable (convert text to numeric if needed)
    treat_mapping = {
        "treat1": 1,
        "treat2": 2,
        "treat3": 3,
        "treat4": 4,
        "treat5": 5,
        "treat6": 6,
    }
    df["treat"] = df["treat"].replace(treat_mapping)
    df["treat"] = pd.to_numeric(df["treat"], errors="coerce")

    # Recode gender
    df["gender"] = df["gender"].replace(GENDER_MAPPINGS)
    df["gender"] = pd.to_numeric(df["gender"], errors="coerce")

    # Recode ideology
    df["leftright_1"] = df["leftright_1"].replace(IDEOLOGY_MAPPINGS)
    df["leftright_1"] = pd.to_numeric(df["leftright_1"], errors="coerce")

    return df

In [ ]:
from src.config import HARASSMENT_TREATMENTS, ANIMAL_TREATMENTS
sample = prepare_analysis_sample(load_main_data())

harassment = sample[sample['treat'].isin(HARASSMENT_TREATMENTS)]
animal = sample[sample['treat'].isin(ANIMAL_TREATMENTS)]

print(f"Sexual harassment sample: {len(harassment)} obs")
print(f"  Treatment 1 (All-Male): {(harassment['treat']==1).sum()}")
print(f"  Treatment 2 (Gender-Balanced): {(harassment['treat']==2).sum()}")
print(f"  Treatment 3 (Quota): {(harassment['treat']==3).sum()}")

print(f"\nAnimal mistreatment sample: {len(animal)} obs")
print(f"  Treatment 4 (All-Male): {(animal['treat']==4).sum()}")
print(f"  Treatment 5 (Gender-Balanced): {(animal['treat']==5).sum()}")
print(f"  Treatment 6 (Quota): {(animal['treat']==6).sum()}")

print(f"\nCountries in harassment sample: {harassment['Country'].nunique()}")
print(f"Countries in animal sample: {animal['Country'].nunique()}")


## 1. Country-Level Data Processing


### 1. Country-Level Data Processing

Load and process 10 country datasets:
- Load raw survey data for each country
- Create factor-analyzed legitimacy indices (omega method)
- Standardize outcomes within country (mean=0, SD=1)
- Filter manipulation check failures and speedsters
- Output aggregated country files

Countries: Argentina, Australia, Brazil, France, Mexico, New Zealand,
Norway, Peru, Portugal, Spain, UK, USA

In [ ]:
def run():
    """Run country processing for all countries and save aggregated datasets."""
    print("Processing country datasets...")

    # Process standard countries
    standard_countries = ["Argentina", "Brazil", "Mexico", "Peru", "NZ", "USA"]
    for country in standard_countries:
        try:
            agg_df = process_standard_country(country)
            # Save aggregated dataset
            output_file = DATA_CONVERTED / f"{country.replace(' ', '')}Agg_python.csv"
            agg_df.to_csv(output_file, index=False)
            print(f"    Saved {output_file.name} ({len(agg_df)} rows)")
        except Exception as e:
            print(f"    ERROR processing {country}: {e}")

    # Process Vignettes_DO countries
    vignettes_countries = ["UK", "Spain", "Portugal"]
    for country in vignettes_countries:
        try:
            agg_df = process_vignettes_do_country(country)
            output_file = DATA_CONVERTED / f"{country}Agg_python.csv"
            agg_df.to_csv(output_file, index=False)
            print(f"    Saved {output_file.name} ({len(agg_df)} rows)")
        except Exception as e:
            print(f"    ERROR processing {country}: {e}")

    print("Country processing complete!")

In [ ]:
result = run()
print(type(result))
if isinstance(result, dict):
    print(list(result.keys()))

## 2. Merged Cross-Country Analysis


### 2. Merged Cross-Country Analysis

Combine country datasets and generate main figures:

**Figure 2:** Main treatment effects (sexual harassment vignettes)
- Substantive legitimacy by council composition
- Procedural legitimacy by council composition
- Pooled 12-country sample with 95% confidence intervals

**Figure 4:** USA vs UK comparison
- Treatment effects separately for two non-quota countries
- Highlights quota penalty in countries without statutory quotas

Treatment conditions:
1. All-male council (8 men)
2. Gender-balanced council (4 men, 4 women)
3. Quota-elected gender-balanced council (4 men, 4 women via quota rule)

In [ ]:
def run():
    """Run merged analysis and generate figures."""
    print("Running merged analysis...")

    # Load merged data
    merged = load_main_data()
    print(f"  Loaded {len(merged)} observations from {merged['Country'].nunique()} countries")

    # Prepare analysis sample
    merged = prepare_analysis_sample(merged)

    # Create figures
    print("  Creating Figure 2...")
    create_figure_2(merged)

    print("  Creating Figure 4...")
    create_figure_4(merged)

    print("Merged analysis complete!")

In [ ]:
result = run()
print(type(result))
if isinstance(result, dict):
    print(list(result.keys()))

## 3. Balance Tests (Randomization Check)


### 3. Balance Tests (Randomization Check)

Verify experimental randomization by comparing treatment groups on
pre-treatment covariates.

**Table SI.4:** Sexual harassment treatments (treats 1-3)
- Gender distribution across treatments
- Ideology (left-right scale) across treatments
- Country distribution across treatments

**Table SI.5:** Animal mistreatment treatments (treats 4-6)
- Same balance checks for non-gendered issue vignettes

Expected result: No significant differences (randomization successful)

## 4. Main Treatment Effects (T-Tests)


### 4. Main Treatment Effects (T-Tests)

Bivariate comparisons of legitimacy outcomes across treatment conditions.

**Table SI.6:** Sexual harassment vignettes (12 countries pooled)
- All-Male vs Gender-Balanced (treats 1 vs 2)
- All-Male vs Quota Gender-Balanced (treats 1 vs 3)
- Gender-Balanced vs Quota Gender-Balanced (treats 2 vs 3) = "quota penalty"
- For both substantive and procedural legitimacy

**Table SI.8:** USA and UK specific effects
- Country-specific treatment effects for two non-quota countries

**Table SI.9:** Animal mistreatment vignettes (9 countries)
- Same treatment comparisons for non-gendered issue

Method: Two-sample t-tests with 95% confidence intervals

## 5. OLS Regressions with Covariates


### 5. OLS Regressions with Covariates

Model-based estimates of treatment effects controlling for covariates
and country fixed effects.

**Table SI.10:** Substantive legitimacy regressions
- Model 1: All-Male vs Gender-Balanced + controls
- Model 2: All-Male vs Quota Gender-Balanced + controls
- Model 3: Gender-Balanced vs Quota Gender-Balanced + controls

**Table SI.11:** Procedural legitimacy regressions
- Same models as SI.10 but for procedural outcome

Controls:
- Gender (respondent)
- Political ideology (leftright_1, 0-10 scale)
- Country fixed effects (11 dummies)

Formula: Outcome ~ treat + gender + leftright_1 + Country

## 6. Robustness Checks (Excluding Political Left)


### 6. Robustness Checks (Excluding Political Left)

Test whether results hold when excluding liberal respondents
(ideology ≤ 4 on 0-10 scale).

**Table SI.12:** Substantive legitimacy (moderates/conservatives only)
- Same three models as SI.10
- Subset: leftright_1 > 4
- Expected: Effects attenuated but still significant

**Table SI.13:** Procedural legitimacy (moderates/conservatives only)
- Same three models as SI.11
- Subset: leftright_1 > 4

Purpose: Address concern that quota support driven by left-leaning
respondents only.